In [0]:
%pip install --upgrade kagglehub

In [0]:
import os
from pathlib import Path
import kagglehub
from pyspark.sql.functions import current_timestamp, lit

In [0]:
KAGGLE_DATASET = "hubertsidorowicz/steam-games-dataset-daily-updates"
VOLUME_PATH = "/Volumes/steam_game_intelligence/bronze/kaggle_data"

CATALOG = "steam_game_intelligence"
SCHEMA = "bronze"
TABLE_NAMES = ["steam_games", "steam_games_review"]

SECRET_SCOPE = "kaggle"
SECRET_KEY = "kaggleAPIKey"

In [0]:
KAGGLE_API_TOKEN = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key=SECRET_KEY
)

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

print("Kaggle authentication configured.")

In [0]:
dataset_path = kagglehub.dataset_download(
    KAGGLE_DATASET,
    output_dir=VOLUME_PATH,
    force_download=True
)

print(f"Dataset downloaded to: {dataset_path}")

In [0]:
from pathlib import Path

files = [
    file
    for file in Path(VOLUME_PATH).rglob("*")
    if file.is_file()
]

for file in files:
    print(file)

In [0]:
games_file = next(
    (
        file
        for file in files
        if file.name.lower() == "steam_games.csv"
    ),
    None
)

review_file = next(
    (
        file
        for file in files
        if file.name.lower() == "steam_games_reviews.csv"
    ),
    None
)

if games_file is None:
    raise FileNotFoundError(
        "Could not find 'steam_games.csv' in the Kaggle dataset."
    )


if review_file is None:
    raise FileNotFoundError(
        "Could not find 'steam_games_reviews' in the Kaggle dataset."
    )

print(f"Reading file: {games_file}")
print(f"Reading file: {review_file}")

In [0]:
games_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .load(str(games_file))
)

review_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .load(str(review_file))
)

In [0]:
games_df = (
    games_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn(
        "_source",
        lit("https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates")
    )
)

review_df = (
    review_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn(
        "_source",
        lit("https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates")
    )
)

In [0]:
print("Schema:")
games_df.printSchema()

print(f"Rows: {games_df.count()}")
print(f"Columns: {len(games_df.columns)}")

display(games_df.limit(20))

In [0]:
print("Schema:")
review_df.printSchema()

print(f"Rows: {review_df.count()}")
print(f"Columns: {len(review_df.columns)}")

display(review_df.limit(20))

In [0]:
GAMES_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAMES[0]}"

(
    games_df
    .write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", True)
    .saveAsTable(GAMES_TABLE)
)

print(f"Table created successfully:")
print(GAMES_TABLE)

In [0]:
REVIEW_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAMES[1]}"

(
    review_df
    .write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", True)
    .saveAsTable(REVIEW_TABLE)
)

print(f"Table created successfully:")
print(REVIEW_TABLE)